# v13 — train 전용 범주형·비선형 실험
목표 홀드아웃 Macro F1 0.60. test/sample 접근을 audit hook으로 차단하며 제출 파일을 생성하지 않습니다. 기존 홀드아웃은 여러 번 관측했으므로 독립 최종 평가가 아닙니다. Python 3.14 + requirements-v13.txt 환경에서 Run All로 선별·학습·검증·pickle 저장을 실행합니다.

In [1]:
import os,sys,re,json,pickle,cloudpickle,hashlib,platform,time,copy,warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.special import softmax
from sklearn.base import BaseEstimator,TransformerMixin,clone
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_selection import chi2,f_classif
from sklearn.preprocessing import StandardScaler,MaxAbsScaler
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import f1_score,accuracy_score,classification_report,confusion_matrix
from lightgbm import LGBMClassifier
from threadpoolctl import threadpool_limits
from IPython.display import display,FileLink
threadpool_limits(2)
SEED=42
forbidden_attempts13=[]
def forbid_inference13(event,args):
 if event=='open' and args and isinstance(args[0],(str,bytes,os.PathLike)) and Path(os.fsdecode(args[0])).name in {'test.csv','sample_submission.csv'}:
  forbidden_attempts13.append(str(args[0]));raise RuntimeError('Inference data access is forbidden in v13')
sys.addaudithook(forbid_inference13)
roots=[Path.cwd(),*Path.cwd().parents]
REPO=next((p for p in roots if (p/'data/train.csv').is_file()),None)
if REPO is None:raise FileNotFoundError('data/train.csv가 필요합니다.')
DATA=REPO/'data';WORK=REPO/'models'/'혜진' if (REPO/'models'/'혜진').is_dir() else REPO
OUT=WORK/'artifacts'/'v13';OUT.mkdir(parents=True,exist_ok=True)
# Inference files and submission directory are intentionally not used.
os.environ['MPLCONFIGDIR']=str(OUT/'.mplconfig')
train=pd.read_csv(DATA/'train.csv')
genes=[c for c in train if c not in ['ID','SUBCLASS']]
assert train.ID.is_unique
y=train.SUBCLASS.to_numpy();labels=np.sort(np.unique(y));yi=np.searchsorted(labels,y);NC=len(labels)
X=train[genes];groups=pd.util.hash_pandas_object(X,index=False).to_numpy()
dev,hold=next(StratifiedGroupKFold(5,shuffle=True,random_state=SEED).split(X,y,groups))
folds=list(StratifiedGroupKFold(3,shuffle=True,random_state=SEED+1).split(X.iloc[dev],y[dev],groups[dev]))
assert not set(groups[dev]).intersection(groups[hold])
for a,b in folds:assert not set(groups[dev[a]]).intersection(groups[dev[b]])
pd.DataFrame({'ID':train.ID,'split':np.where(np.isin(np.arange(len(train)),hold),'holdout','development'),'group':groups.astype(str)}).to_csv(OUT/'split_manifest.csv',index=False)
print('TRAIN ONLY',train.shape,'Dev/holdout',len(dev),len(hold),'OUT',OUT,flush=True)

from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier


TRAIN ONLY (6201, 4386) Dev/holdout 4960 1241 OUT /Users/mac/project/cancer_project/artifacts/v13


In [2]:
GROUPS = {
 'p53_cellcycle': 'TP53 RB1 CDKN2A CDKN2B CDK4 CDK6 CCND1 ATM CHEK2'.split(),
 'rtk_ras_mapk': 'EGFR ERBB2 ERBB3 KRAS NRAS HRAS BRAF NF1 MAP2K1 MAP2K2 KIT PDGFRA FGFR2 FGFR3'.split(),
 'pi3k_mtor': 'PIK3CA PIK3R1 PTEN AKT1 MTOR TSC1 TSC2 STK11'.split(),
 'dna_repair': 'BRCA1 BRCA2 PALB2 ATM ATR CHEK2 MLH1 MSH2 MSH6 PMS2 POLE POLD1'.split(),
 'chromatin': 'ARID1A ARID1B ARID2 PBRM1 SMARCA4 BAP1 SETD2 KMT2A KMT2C KMT2D ATRX'.split(),
 'wnt': 'APC CTNNB1 AXIN1 AXIN2 RNF43'.split(),
 'metabolism': 'IDH1 IDH2 SDHB SDHC SDHD FH VHL'.split(),
}

def kind(token):
 if 'fs' in token.lower(): return 'frameshift'
 if '*' in token: return 'stop'
 if 'del' in token.lower(): return 'deletion'
 if 'ins' in token.lower(): return 'insertion'
 m = re.fullmatch(r'([A-Z])(\d+)([A-Z])', token)
 if m: return 'synonymous' if m[1] == m[3] else 'substitution'
 return 'other'

class MutationFeatures(BaseEstimator, TransformerMixin):
 def __init__(self, mode='full', min_df=3):
  self.mode = mode
  self.min_df = min_df

 def dictionaries(self, X):
  genes = list(X.columns)
  for row in X.itertuples(index=False, name=None):
   d = {}; counts = {}; burden = 0; events = 0; positions = []
   changed = set()
   for gene, value in zip(genes, row):
    if value is None or str(value) in ('nan', '', '<NA>'):
     d['missing:' + gene] = 1; continue
    if value == 'WT': continue
    changed.add(gene); burden += 1; d['gene:' + gene] = 1
    if self.mode == 'binary': continue
    tokens = str(value).split()
    for raw_token in tokens:
     token = re.sub(r'^([A-Z]\d+)X$', r'\1*', raw_token)
     events += 1; typ = kind(token); counts[typ] = counts.get(typ, 0) + 1
     d['variant:' + gene + ':' + token] = 1
     d['type:' + gene + ':' + typ] = 1
     m = re.search(r'\d+', token)
     if m:
      position = int(m[0]); positions.append(position)
      d['position:' + gene + ':' + str(position)] = 1
   if self.mode == 'full':
    d['numeric:log_mutated_genes'] = np.log1p(burden)
    d['numeric:log_events'] = np.log1p(events)
    d['numeric:multi_event_excess'] = np.log1p(max(0, events-burden))
    for typ in ['frameshift','stop','deletion','insertion','synonymous','substitution','other']:
     d['numeric:log_' + typ] = np.log1p(counts.get(typ,0))
     d['numeric:fraction_' + typ] = counts.get(typ,0)/max(events,1)
    d['numeric:log_mean_position'] = np.log1p(np.mean(positions)) if positions else 0.
    for name, members in GROUPS.items():
     d['module:' + name] = np.log1p(len(changed.intersection(members)))
   yield d

 def fit(self, X, y=None):
  self.genes_ = list(X.columns)
  self.vectorizer_ = DictVectorizer(dtype=np.float32)
  matrix = self.vectorizer_.fit_transform(self.dictionaries(X))
  names = self.vectorizer_.get_feature_names_out()
  self.keep_ = (np.asarray((matrix != 0).sum(axis=0)).ravel() >= self.min_df) | np.array([s.startswith(('numeric:', 'module:')) for s in names])
  self.feature_names_ = names[self.keep_]
  return self

 def transform(self, X):
  missing = set(self.genes_) - set(X.columns)
  if missing: raise ValueError('Missing gene columns: ' + str(sorted(missing)[:10]))
  return sparse.csr_matrix(self.vectorizer_.transform(self.dictionaries(X[self.genes_]))[:, self.keep_])

 def get_feature_names_out(self, input_features=None): return self.feature_names_
def canon(t):
 t=t.strip().removeprefix('p.');t=re.sub(r'(?<=\d)X$','*',t)
 return re.sub(r'^([A-Z]\d+)(?:[A-Z*]+)?fs.*$',r'\1fs',t)
def aa_change(t):
 m=re.fullmatch(r'([A-Z])\d+([A-Z*])',t)
 if m:return m[1]+'>'+m[2]
 m=re.match(r'([A-Z])\d+',t)
 if m:
  for suffix in ['fs','del','ins']:
   if suffix in t:return m[1]+'>'+suffix
 return None

def parse_events(frame):
 names=list(frame.columns);result=[]
 for row in frame.itertuples(index=False,name=None):
  result.append([(g,tuple(sorted(set(canon(t) for t in v.split())))) for g,v in zip(names,row) if isinstance(v,str) and v!='WT'])
 return result

class EvidenceEncoder:
 def __init__(self,unknown=1.,silent=1.,support=2):self.unknown=unknown;self.silent=silent;self.support=support
 def fit(self,events):
  self.token_df={};self.change_df={}
  if self.unknown<1:
   tokens=Counter();changes=Counter()
   for row in events:
    for g,ts in row:
     tokens.update((g,t) for t in ts)
     changes.update(set((g,a) for t in ts if (a:=aa_change(t)) is not None))
   self.token_df={k:v for k,v in tokens.items() if v>=self.support}
   self.change_df={k:v for k,v in changes.items() if v>=self.support}
  return self
 def transform(self,events,training=False,exclude_events=None):
  result=[]
  for row_number,row in enumerate(events):
   own=exclude_events[row_number] if exclude_events is not None else (row if training else [])
   own_tokens={(gg,tt) for gg,tts in own for tt in tts}
   own_changes={(gg,ac) for gg,tts in own for tt in tts if (ac:=aa_change(tt)) is not None}
   d={};gene_weights={};type_sums=Counter();non_syn=0.;trunc=0.;multikind=0
   for g,ts in row:
    types={};gene=0.;known_gene=False
    for t in ts:
     typ=kind(t);a=aa_change(t)
     known=self.token_df.get((g,t),0)-int((g,t) in own_tokens)>=self.support
     known_aa=a is not None and self.change_df.get((g,a),0)-int((g,a) in own_changes)>=self.support
     if typ=='synonymous':w=self.silent
     elif known or typ in ['stop','frameshift','deletion','insertion']:w=1.
     elif known_aa:w=max(self.unknown,.75)
     else:w=self.unknown
     if w<=0:continue
     gene=max(gene,w);types[typ]=max(types.get(typ,0),w)
     d['variant:'+g+':'+t]=max(d.get('variant:'+g+':'+t,0),w)
     m=re.search(r'\d+',t)
     if m:d['position:'+g+':'+m[0]]=max(d.get('position:'+g+':'+m[0],0),w)
     if a:d['aa:'+g+':'+a]=max(d.get('aa:'+g+':'+a,0),w)
     if known and typ!='synonymous':known_gene=True
    if gene<=0:continue
    d['gene:'+g]=gene;gene_weights[g]=gene
    for typ,w in types.items():d['type:'+g+':'+typ]=w;type_sums[typ]+=w
    if known_gene:d['supported:'+g]=1.
    non_syn+=max([w for k,w in types.items() if k!='synonymous'] or [0.])
    trunc+=max([w for k,w in types.items() if k in ['stop','frameshift','deletion','insertion']] or [0.])
    multikind+=len(types)>1
   n=len(row)
   d['numeric:log_raw_genes']=np.log1p(n)
   d['numeric:log_effective_genes']=np.log1p(sum(gene_weights.values()))
   d['numeric:log_nonsyn_genes']=np.log1p(non_syn)
   d['numeric:log_trunc_genes']=np.log1p(trunc)
   d['numeric:log_multikind_genes']=np.log1p(multikind)
   for typ in ['synonymous','substitution','stop','frameshift','deletion','insertion','other']:
    d['numeric:gene_fraction_'+typ]=type_sums[typ]/max(n,1)
   for name,gs in GROUPS.items():d['module:'+name]=np.log1p(sum(gene_weights.get(g,0) for g in gs))
   result.append(d)
  return result

events=parse_events(X)
legacy_records=list(MutationFeatures(mode='full').dictionaries(X))
assert canon('R213X')=='R213*' and canon('K16Rfs')=='K16fs'
assert aa_change('R132H')==aa_change('R140H')=='R>H'
print('Parsed rows:',len(events),'Normalization examples passed')


Parsed rows: 6201 Normalization examples passed


In [3]:
def relevance_scores(a,target,method):
 base=method.removesuffix('_full')
 if base=='embedded':
  scaled=csr32(MaxAbsScaler().fit_transform(a))
  fitted=OneVsRestClassifier(LogisticRegression(solver='liblinear',l1_ratio=1,C=.2,class_weight='balanced',max_iter=2000),n_jobs=1).fit(scaled,target)
  return np.max(np.abs(np.vstack([e.coef_[0] for e in fitted.estimators_])),axis=0)
 return (chi2(a,target) if base=='chi2' else f_classif(a,target))[0]

def csr32(a):
 b=sparse.csr_matrix(a,dtype=np.float32)
 if b.nnz>=np.iinfo(np.int32).max:raise ValueError('Sparse matrix too large for tree index type')
 b.indices=b.indices.astype(np.int32,copy=False);b.indptr=b.indptr.astype(np.int32,copy=False)
 return b

class Space:
 def fit(self,rs):
  self.vec=DictVectorizer(dtype=np.float32);a=self.vec.fit_transform(rs)
  names=self.vec.get_feature_names_out();df=np.asarray((a!=0).sum(axis=0)).ravel()
  keep=(df>=3)|np.array([n.startswith(('numeric:','module:')) for n in names])
  self.raw=np.flatnonzero(keep);self.names=names[keep]
  return csr32(a[:,self.raw])
 def transform(self,rs):return csr32(self.vec.transform(rs)[:,self.raw])
 def compact_vec(self,indices):
  v=copy.deepcopy(self.vec);mask=np.zeros(len(v.feature_names_),bool);mask[self.raw[np.sort(indices)]]=True
  v.restrict(mask);return v

class Selector:
 def __init__(self,a,target,names):
  self.a=a;self.y=target;self.names=names;self.cache={};self.corr={};self.scores={};self.audit=[];self.examples=[]
 def order(self,method,threshold):
  key=(method,threshold)
  if key in self.cache:return self.cache[key]
  if method not in self.scores:
   with warnings.catch_warnings():
    warnings.simplefilter('ignore',RuntimeWarning);score=relevance_scores(self.a,self.y,method)
   score=np.nan_to_num(score,nan=0,posinf=np.finfo(np.float32).max,neginf=0)
   if not method.endswith('_full'):score[np.array([n.startswith('numeric:') for n in self.names])]=0
   self.scores[method]=score
  score=self.scores[method]
  if method not in self.corr:
   pool=np.argsort(-score,kind='stable')[:min(4096,int((score>0).sum()))]
   z=self.a[:,pool];mean=np.asarray(z.mean(axis=0)).ravel();var=np.asarray(z.multiply(z).mean(axis=0)).ravel()-mean**2
   sd=np.sqrt(np.maximum(var,1e-12));corr=(z.T@z).toarray()/z.shape[0]-np.outer(mean,mean)
   corr/=np.outer(sd,sd);np.clip(corr,-1,1,out=corr)
   self.corr[method]=(pool,corr)
  pool,corr=self.corr[method];chosen=[]
  for i in range(len(pool)):
   if chosen:
    correlations=np.abs(corr[i,chosen]);j=int(np.argmax(correlations))
    if correlations[j]>=threshold:
     if len(self.examples)<100:self.examples.append({'method':method,'threshold':threshold,'kept':self.names[pool[chosen[j]]],'discarded':self.names[pool[i]],'correlation':float(corr[i,chosen[j]])})
     continue
   chosen.append(i)
   if len(chosen)>=2048:break
  order=pool[chosen];self.cache[key]=order
  self.audit.append({'method':method,'threshold':threshold,'association_pool':len(pool),'remaining_up_to_2048':len(order)})
  return order
 def select(self,method,threshold,k):return np.sort(self.order(method,threshold)[:k])

class CompactPredictor:
 def __init__(self,space,cols,est):
  self.vec=space.compact_vec(cols);self.names=self.vec.get_feature_names_out();self.est=est
  self.direct_inputs=len(self.names);self.raw_inputs=len(self.names);self.kind='compact'
 def predict(self,rs):return self.est.predict_proba(csr32(self.vec.transform(rs)))

def background_frame(frame,training_frame,seed,count):
 rng=np.random.default_rng(seed);out=frame.copy();arr=out.to_numpy(copy=True)
 rate=(training_frame.notna()&training_frame.ne('WT')).mean().to_numpy();prob=rate/rate.sum();idx=np.flatnonzero(prob>0)
 for r in range(len(arr)):
  n=min(len(idx),int(rng.poisson(count)))
  for c in rng.choice(len(prob),size=n,replace=False,p=prob):
   token='A1A' if rng.random()<.5 else 'A1V'
   arr[r,c]=token if not isinstance(arr[r,c],str) or arr[r,c]=='WT' else arr[r,c]+' '+token
 return pd.DataFrame(arr,columns=frame.columns,index=frame.index)
class ColumnPost:
 def __init__(self,mode='none'):self.mode=mode
 def fit(self,a,names):
  self.raw_names=np.array(names);self.names=np.array(names)
  if self.mode=='maxabs':self.scaler=MaxAbsScaler().fit(a)
  if self.mode=='block':
   self.binary=np.flatnonzero([not n.startswith(('numeric:','module:')) for n in names])
   self.summary=np.flatnonzero([n.startswith(('numeric:','module:')) for n in names])
   self.tf=TfidfTransformer(norm='l2',use_idf=True).fit(a[:,self.binary])
   self.scaler=MaxAbsScaler().fit(a[:,self.summary])
   self.names=np.concatenate([self.raw_names[self.binary],self.raw_names[self.summary]])
  return self
 def transform(self,a):
  if self.mode=='maxabs':return csr32(self.scaler.transform(a))
  if self.mode=='block':return csr32(sparse.hstack([self.tf.transform(a[:,self.binary]),self.scaler.transform(a[:,self.summary])],format='csr'))
  return csr32(a)

class Member:
 def __init__(self,name,view,encoder,space,cols,post,est):
  self.name=name;self.view=view;self.encoder=encoder;self.vec=space.compact_vec(cols);self.post=post;self.est=est
  self.raw_names=self.vec.get_feature_names_out();self.names=post.names
 def predict_rows(self,rs):return self.est.predict_proba(self.post.transform(csr32(self.vec.transform(rs))))

VIEWS={'raw':(1.,1.),'syn_down':(1.,.25),'gated':(.25,1.),'gated_syn_down':(.25,.25),'gated_nonsyn':(.25,0.)}
CONFIGS={}
for view in VIEWS:
 for k in [1024,2048]:
  for c in [.1,.5]:CONFIGS[f'{view}_lr_k{k}_C{c}']=dict(view=view,algo='lr',k=k,C=c,post='none',method='chi2_full')
  CONFIGS[f'{view}_lgb_k{k}']=dict(view=view,algo='lgb',k=k,post='none',method='f_classif_full')
for view in ['raw','gated_syn_down']:
 for depth in [3,5]:CONFIGS[f'{view}_xgb_d{depth}']=dict(view=view,algo='xgb',k=2048,depth=depth,post='none',method='f_classif_full')
 for alpha in [1,10]:CONFIGS[f'{view}_mlp_a{alpha}']=dict(view=view,algo='mlp',k=1024,alpha=alpha,post='maxabs',method='chi2_full')
 for c in [1,4]:CONFIGS[f'{view}_blocklr_C{c}']=dict(view=view,algo='lr',k=2048,C=c,post='block',method='chi2_full')
 CONFIGS[f'{view}_extra']=dict(view=view,algo='extra',k=1024,post='none',method='f_classif_full')
(OUT/'configurations.json').write_text(json.dumps(CONFIGS,indent=2))

def estimator(cfg):
 algo=cfg['algo']
 if algo=='lr':return LogisticRegression(C=cfg['C'],class_weight='balanced',max_iter=2000)
 if algo=='lgb':return LGBMClassifier(n_estimators=300,learning_rate=.04,num_leaves=15,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=SEED)
 if algo=='xgb':return XGBClassifier(n_estimators=300,max_depth=cfg['depth'],learning_rate=.04,min_child_weight=3,reg_lambda=10,subsample=.85,colsample_bytree=.85,max_bin=63,tree_method='hist',objective='multi:softprob',eval_metric='mlogloss',n_jobs=2,random_state=SEED)
 if algo=='mlp':return MLPClassifier(hidden_layer_sizes=(128,),activation='relu',alpha=cfg['alpha'],learning_rate_init=.001,batch_size=128,max_iter=250,tol=1e-4,n_iter_no_change=15,early_stopping=False,random_state=SEED)
 return ExtraTreesClassifier(n_estimators=250,max_depth=20,min_samples_leaf=3,max_features='sqrt',class_weight='balanced',n_jobs=2,random_state=SEED)

def prepare_view(idx,view):
 ev=[events[i] for i in idx];enc=EvidenceEncoder(*VIEWS[view]).fit(ev)
 rs=enc.transform(ev,training=True);space=Space();a=space.fit(rs);selector=Selector(a,yi[idx],space.names)
 return enc,space,a,selector

def train_new(name,idx,cache):
 cfg=CONFIGS[name];view=cfg['view']
 if view not in cache:cache[view]=prepare_view(idx,view)
 enc,space,a,selector=cache[view];cols=selector.select(cfg['method'],.98,cfg['k'])
 post=ColumnPost(cfg['post']).fit(a[:,cols],space.names[cols]);tx=post.transform(a[:,cols]);target=yi[idx]
 row_weights=np.ones(len(idx),dtype=float)
 if cfg.get('augment',False):
  if '_aug_events' not in cache:
   cache['_aug_events']=parse_events(rare_background(X.iloc[idx],[events[i] for i in idx],SEED+777,count=10))
  aug_rs=enc.transform(cache['_aug_events'],exclude_events=[events[i] for i in idx])
  aug_x=post.transform(space.transform(aug_rs)[:,cols]);tx=csr32(sparse.vstack([tx,aug_x],format='csr'))
  target=np.concatenate([target,target]);row_weights=np.concatenate([row_weights,.5*row_weights])
 est=estimator(cfg)
 if cfg['algo'] in ['xgb','mlp']:est.fit(tx,target,sample_weight=compute_sample_weight('balanced',target)*row_weights)
 elif cfg.get('augment',False):est.fit(tx,target,sample_weight=row_weights)
 else:est.fit(tx,target)
 return Member(name,view,enc,space,cols,post,est)

LEGACY={
 'legacy_lr':dict(k=2048,method='chi2',algo='lr'),
 'legacy_lgb':dict(k=2048,method='f_classif_full',algo='lgb'),
 'legacy_extra':dict(k=1024,method='f_classif_full',algo='extra')}
def train_legacy(idx):
 space=Space();a=space.fit([legacy_records[i] for i in idx]);sel=Selector(a,yi[idx],space.names);result=[]
 for name,cfg in LEGACY.items():
  cols=sel.select(cfg['method'],.95,cfg['k'])
  if cfg['algo']=='lr':est=LogisticRegression(C=.1,class_weight='balanced',max_iter=2000)
  elif cfg['algo']=='lgb':est=LGBMClassifier(n_estimators=250,learning_rate=.05,num_leaves=15,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=SEED)
  else:est=ExtraTreesClassifier(n_estimators=250,max_depth=20,min_samples_leaf=3,max_features='sqrt',class_weight='balanced',n_jobs=2,random_state=SEED)
  est.fit(a[:,cols],yi[idx]);post=ColumnPost().fit(a[:,cols],space.names[cols]);result.append(Member(name,'legacy',None,space,cols,post,est))
 return result

def duplicate_frame(frame):
 arr=frame.to_numpy(copy=True)
 for i,row in enumerate(arr):
  for j,v in enumerate(row):
   if isinstance(v,str) and v!='WT':arr[i,j]=v+' '+v
 return pd.DataFrame(arr,columns=frame.columns,index=frame.index)

def rare_background(frame,tr_events,seed,count=20):
 rng=np.random.default_rng(seed);df=Counter((g,t) for row in tr_events for g,ts in row for t in ts)
 syn=[];sub=[]
 for (g,t),n in df.items():
  if n>2:continue
  if kind(t)=='synonymous':syn.append((g,t))
  elif kind(t)=='substitution':sub.append((g,t))
 arr=frame.to_numpy(copy=True);lookup={g:i for i,g in enumerate(frame.columns)}
 for row in arr:
  for _ in range(int(rng.poisson(count))):
   pool=syn if rng.random()<.5 else sub
   g,t=pool[int(rng.integers(len(pool)))];j=lookup[g]
   row[j]=t if not isinstance(row[j],str) or row[j]=='WT' else row[j]+' '+t
 return pd.DataFrame(arr,columns=frame.columns,index=frame.index)

def eval_views(idx,tr_idx,seed):
 clean=X.iloc[idx];noise=rare_background(clean,[events[i] for i in tr_idx],seed);dup=duplicate_frame(clean)
 frames=[clean,noise,dup]
 return frames,[parse_events(f) for f in frames],[list(MutationFeatures(mode='full').dictionaries(f)) for f in frames]

def predict_members(members,weights,frame):
 ev=None;old=None;cache={};p=None
 for m,w in zip(members,weights):
  if m.view=='legacy':
   if old is None:old=list(MutationFeatures(mode='full').dictionaries(frame[genes]))
   rs=old
  else:
   if ev is None:ev=parse_events(frame[genes])
   if m.view not in cache:cache[m.view]=m.encoder.transform(ev)
   rs=cache[m.view]
  q=m.predict_rows(rs);p=q*w if p is None else p+q*w
 return p
print('Novel settings:',len(CONFIGS),'Views:',VIEWS)

# CV-guided targeted refinement: regularization, support threshold, and rare-variant augmentation.
VIEWS['gated_syn_down_s3']=(.25,.25,3)
CONFIGS.update({'raw_mlp_a0.01': {'view': 'raw', 'algo': 'mlp', 'k': 1024, 'alpha': 0.01, 'post': 'maxabs', 'method': 'chi2_full'}, 'raw_mlp_a0.1': {'view': 'raw', 'algo': 'mlp', 'k': 1024, 'alpha': 0.1, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_mlp_a0.01': {'view': 'gated_syn_down', 'algo': 'mlp', 'k': 1024, 'alpha': 0.01, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_mlp_a0.1': {'view': 'gated_syn_down', 'algo': 'mlp', 'k': 1024, 'alpha': 0.1, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k1024_C0.1': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 1024, 'C': 0.1, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k1024_C0.5': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 1024, 'C': 0.5, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lgb_k1024': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 1024, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_lr_k2048_C0.1': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 2048, 'C': 0.1, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k2048_C0.5': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 2048, 'C': 0.5, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lgb_k2048': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_xgb_d3': {'view': 'gated_syn_down_s3', 'algo': 'xgb', 'k': 2048, 'depth': 3, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_xgb_d5': {'view': 'gated_syn_down_s3', 'algo': 'xgb', 'k': 2048, 'depth': 5, 'post': 'none', 'method': 'f_classif_full'}, 'raw_aug_lgb': {'view': 'raw', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full', 'augment': True}, 'raw_aug_xgb_d5': {'view': 'raw', 'algo': 'xgb', 'k': 2048, 'depth': 5, 'post': 'none', 'method': 'f_classif_full', 'augment': True}, 'gated_s3_aug_lgb': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full', 'augment': True}})
(OUT/'configurations.json').write_text(json.dumps(CONFIGS,indent=2))
print('Total new configurations after refinement:',len(CONFIGS))


Novel settings: 44 Views: {'raw': (1.0, 1.0), 'syn_down': (1.0, 0.25), 'gated': (0.25, 1.0), 'gated_syn_down': (0.25, 0.25), 'gated_nonsyn': (0.25, 0.0)}
Total new configurations after refinement: 59


In [4]:
# Freeze the previous version; do not repeat its 59-setting search.
BASE_CONFIGS=copy.deepcopy(CONFIGS)
BASE_PLAN=[('v4_reference',.25),('gated_s3_aug_lgb',.1875),('raw_aug_xgb_d5',.1875),('raw_lr_k2048_C0.1',.1875),('gated_syn_down_mlp_a1',.1875)]

def fit_base(idx):
 cache={};ms=[];ws=[]
 for name,w in BASE_PLAN:
  if name=='v4_reference':ms.extend(train_legacy(idx));ws.extend([w/3]*3)
  else:ms.append(train_new(name,idx,cache));ws.append(w)
 return ms,np.array(ws)


In [5]:
class NewSelector:
 def __init__(self,a,target,names):self.a=a;self.y=target;self.names=names;self.cache={}
 def select(self,method,k):
  if (method,k) in self.cache:return self.cache[(method,k)]
  numeric=np.array([s.startswith(('numeric:','module:')) for s in self.names])
  with warnings.catch_warnings():
   warnings.simplefilter('ignore',RuntimeWarning)
   global_score=np.nan_to_num(f_classif(self.a,self.y)[0],nan=0,posinf=0,neginf=0)
   if method=='balanced':
    orders=[]
    for c in range(NC):
     yes=self.y==c
     score=np.nan_to_num(chi2(self.a,yes)[0],nan=0,posinf=0,neginf=0)
     high=np.asarray(self.a[yes].mean(axis=0)).ravel()>np.asarray(self.a[~yes].mean(axis=0)).ravel()
     score[~high|numeric]=0
     orders.append(np.argsort(-score,kind='stable')[:min(k,int((score>0).sum()))])
    order=[]
    for rank in range(max(map(len,orders))):
     for o in orders:
      if rank<len(o):order.append(int(o[rank]))
    order+=list(np.argsort(-global_score,kind='stable'))
   else:order=list(np.argsort(-global_score,kind='stable'))
  z=self.a.tocsc();chosen=list(np.flatnonzero(numeric));seen=set(chosen);signatures=set()
  for j in order:
   j=int(j)
   if j in seen:continue
   seen.add(j)
   if global_score[j]<=0:continue
   lo,hi=z.indptr[j:j+2]
   sig=hashlib.sha256(z.indices[lo:hi].tobytes()+z.data[lo:hi].tobytes()).digest()
   if sig in signatures:continue
   signatures.add(sig);chosen.append(j)
   if len(chosen)>=k:break
  self.cache[(method,k)]=np.sort(chosen);return self.cache[(method,k)]

class SpecificityPost:
 def fit(self,a,names,target):
  self.names=np.array(names);self.binary=np.flatnonzero([not n.startswith(('numeric:','module:')) for n in names]);self.summary=np.flatnonzero([n.startswith(('numeric:','module:')) for n in names])
  z=(a[:,self.binary]!=0).astype(np.float32)
  counts=np.vstack([np.asarray(z[target==c].sum(axis=0)).ravel() for c in range(NC)])
  prior=np.bincount(target,minlength=NC)/len(target);freq=counts.sum(axis=0)
  posterior=(counts+10*prior[:,None])/(freq+10)
  kl=(posterior*np.log(np.maximum(posterior,1e-12)/prior[:,None])).sum(axis=0)
  self.weights=(.25+np.sqrt(np.maximum(kl,0))*freq/(freq+20)).astype(np.float32)
  self.scaler=MaxAbsScaler().fit(a[:,self.summary]);return self
 def transform(self,a):
  from sklearn.preprocessing import normalize
  block=normalize(a[:,self.binary].multiply(self.weights),norm='l2')
  return csr32(sparse.hstack([block,self.scaler.transform(a[:,self.summary])],format='csr'))

NEW={}
for view in ['raw','gated_syn_down_s3']:
 for method in ['global','balanced']:
  NEW[f'{view}_{method}_lgb4096']=dict(view=view,algo='lgb',k=4096,method=method)
for c in [.1,.5]:NEW[f'balanced_lr_C{c}']=dict(view='raw',algo='lr',k=2048,method='balanced',C=c)
for c in [1.,4.]:NEW[f'specificity_lr_C{c}']=dict(view='raw',algo='lr',k=4096,method='balanced',C=c,specificity=True)

def fit_new6(name,idx,cache):
 cfg=NEW[name];view=cfg['view']
 if view not in cache:
  enc,space,a,_=prepare_view(idx,view);cache[view]=(enc,space,a,NewSelector(a,yi[idx],space.names))
 enc,space,a,sel=cache[view];cols=sel.select(cfg['method'],cfg['k']);target=yi[idx]
 if cfg.get('specificity'):post=SpecificityPost().fit(a[:,cols],space.names[cols],target)
 else:post=ColumnPost().fit(a[:,cols],space.names[cols])
 est=estimator(cfg);tx=post.transform(a[:,cols])
 if cfg.get('augment'):
  if '_aug' not in cache:cache['_aug']=parse_events(rare_background(X.iloc[idx],[events[i] for i in idx],SEED+777,count=10))
  ar=enc.transform(cache['_aug'],exclude_events=[events[i] for i in idx]);ax=post.transform(space.transform(ar)[:,cols])
  tx=csr32(sparse.vstack([tx,ax],format='csr'));target=np.concatenate([target,target]);rw=np.r_[np.ones(len(idx)),np.full(len(idx),.5)]
 else:rw=np.ones(len(idx))
 if cfg['algo']=='xgb':est.fit(tx,target,sample_weight=compute_sample_weight('balanced',target)*rw)
 elif cfg.get('augment'):est.fit(tx,target,sample_weight=rw)
 else:est.fit(tx,target)
 m=Member(name,view,enc,space,cols,post,est)
 # Column order of a specificity block changes, but raw_names always describe the vectorizer input.
 if cfg.get('specificity'):m.names=np.concatenate([m.raw_names[post.binary],m.raw_names[post.summary]])
 return m
(OUT/'new_configurations.json').write_text(json.dumps(NEW,indent=2))
print('New configurations',len(NEW))

NEW.update({'gated_s3_aug_global_lgb4096': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 4096, 'method': 'global', 'augment': True}, 'gated_s3_aug_balanced_lgb4096': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 4096, 'method': 'balanced', 'augment': True}, 'raw_aug_global_xgb4096_d3': {'view': 'raw', 'algo': 'xgb', 'k': 4096, 'method': 'global', 'depth': 3, 'augment': True}, 'raw_aug_global_xgb4096_d5': {'view': 'raw', 'algo': 'xgb', 'k': 4096, 'method': 'global', 'depth': 5, 'augment': True}})
(OUT/'new_configurations.json').write_text(json.dumps(NEW,indent=2))

print('Total new configurations including augmentation extension:',len(NEW))


New configurations 8
Total new configurations including augmentation extension: 12


In [6]:
def fit_plan6(idx,plan,basefit=None):
 ms=[];ws=[];cache={}
 for name,w in plan:
  if name=='v5_reference':
   old,ow=basefit if basefit is not None else fit_base(idx)
   ms.extend(old);ws.extend(ow*w)
  else:ms.append(fit_new6(name,idx,cache));ws.append(w)
 return ms,np.array(ws)

V6_PLAN=[('v5_reference',.75),('gated_s3_aug_global_lgb4096',.25)]
def fit_v6(idx):return fit_plan6(idx,V6_PLAN)


In [7]:
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.feature_selection import SelectKBest

def expand10(raw):
 t=canon(raw);m=re.fullmatch(r'([0-9]+)_([0-9]+)([A-Z]+)>([A-Z]+)',t)
 if m and int(m[2])-int(m[1])+1==len(m[3])==len(m[4]):
  return tuple(f'{a}{int(m[1])+j}{b}' for j,(a,b) in enumerate(zip(m[3],m[4])))
 return (t,)

def kind10(t):
 if 'fs' in t:return 'frameshift'
 if t.endswith('*'):return 'stop'
 if 'delins' in t:return 'delins'
 if 'del' in t:return 'deletion'
 if 'ins' in t:return 'insertion'
 m=re.fullmatch(r'([A-Z])[0-9]+([A-Z])',t)
 if m:return 'synonymous' if m[1]==m[2] else 'substitution'
 return 'other'

def packet10(frame,ev=None):
 if ev is None:ev=parse_events(frame[genes])
 resolved=[[(g,tuple(sorted({s for t in ts for s in expand10(t)}))) for g,ts in row] for row in ev]
 missing=frame[genes].isna().to_numpy()
 return dict(events=resolved,missing=missing)

def records10(packet,representation,numeric=True):
 rs=[]
 for i,row in enumerate(packet['events']):
  d={};syn=nonsyn=structural=unparsed=0
  for g,ts in row:
   d['gene:'+g]=1.;types={kind10(t) for t in ts}
   sy='synonymous' in types;ns=bool(types.intersection({'substitution','stop','frameshift','deletion','insertion','delins'}));st=bool(types.intersection({'stop','frameshift','deletion','insertion','delins'}));ot='other' in types
   syn+=sy;nonsyn+=ns;structural+=st;unparsed+=ot
   if representation=='type':
    for typ in types:d['type:'+g+':'+typ]=1.
   elif representation=='functional':
    for flag,family in [(sy,'syn_gene'),(ns,'nonsyn_gene'),(st,'structural_gene'),(ot,'unparsed_gene')]:
     if flag:d[family+':'+g]=1.
  if numeric:
   d['numeric:log_observed_mutated_genes']=np.log1p(len(row));d['numeric:log_missing_genes']=np.log1p(int(packet['missing'][i].sum()))
   if representation!='gene':
    for title,value in [('syn',syn),('nonsyn',nonsyn),('structural',structural),('unparsed',unparsed)]:
     d['numeric:log_'+title+'_genes']=np.log1p(value);d['numeric:fraction_'+title+'_genes']=value/max(len(row),1)
  rs.append(d)
 return rs

def mask_packet10(packet,probability,seed):
 rng=np.random.default_rng(seed);out=[];missing=packet['missing'].copy();gpos={g:j for j,g in enumerate(genes)}
 for i,row in enumerate(packet['events']):
  keep=[]
  for g,ts in row:
   if rng.random()<probability:missing[i,gpos[g]]=True
   else:keep.append((g,ts))
  out.append(keep)
 return dict(events=out,missing=missing)

def select_cols10(a,target,k):
 if k is None or a.shape[1]<=k:return np.arange(a.shape[1])
 with warnings.catch_warnings():
  warnings.simplefilter('ignore',RuntimeWarning);scores=np.nan_to_num(chi2(a,target)[0],nan=0,posinf=0,neginf=0)
 return np.sort(np.argsort(-scores,kind='stable')[:k])

class MissingBernoulli10:
 def __init__(self,alpha=.1,pooled=False,fit_prior=False,k=None):self.alpha=alpha;self.pooled=pooled;self.fit_prior=fit_prior;self.k=k
 def fit(self,a,target,missing,sample_weight=None):
  self.cols=select_cols10(a,target,self.k);a=csr32(a[:,self.cols]);missing=csr32(missing[:,self.cols]);w=np.ones(len(target)) if sample_weight is None else np.asarray(sample_weight)
  self.classes_=np.unique(target);nclass=np.array([w[target==c].sum() for c in self.classes_])
  counts=np.vstack([np.asarray(a[target==c].multiply(w[target==c,None]).sum(axis=0)).ravel() for c in self.classes_])
  absent=np.vstack([np.asarray(missing[target==c].multiply(w[target==c,None]).sum(axis=0)).ravel() for c in self.classes_])
  observed=nclass[:,None]-absent
  if self.pooled:
   denominator=w.sum()-np.asarray(missing.multiply(w[:,None]).sum(axis=0)).ravel()
   prior=np.clip(np.asarray(a.multiply(w[:,None]).sum(axis=0)).ravel()/np.maximum(denominator,1e-9),1e-7,1-1e-7)
   prob=(counts+self.alpha*prior)/(observed+self.alpha)
  else:prob=(counts+self.alpha)/(observed+2*self.alpha)
  prob=np.clip(prob,1e-7,1-1e-7);self.log0=np.log1p(-prob);self.logit=np.log(prob)-self.log0
  prior=nclass/nclass.sum() if self.fit_prior else np.full(len(nclass),1/len(nclass));self.logprior=np.log(prior)
  self.coef_=self.logit;return self
 def predict_proba(self,a,missing):
  a=csr32(a[:,self.cols]);missing=csr32(missing[:,self.cols])
  score=np.asarray(a@self.logit.T)+self.log0.sum(axis=1)+self.logprior-np.asarray(missing@self.log0.T)
  return softmax(score,axis=1)

class GroupSigmoidSVM10:
 def __init__(self,C=.1,k=2048,balanced_calibration=False):self.C=C;self.k=k;self.balanced_calibration=balanced_calibration
 def fit(self,a,target,profile_groups,sample_weight):
  self.classes_=np.unique(target);oof=np.zeros((len(target),len(self.classes_)),float)
  for tr,va in StratifiedGroupKFold(3,shuffle=True,random_state=310).split(a,target,profile_groups):
   assert not set(profile_groups[tr]).intersection(profile_groups[va])
   cols=select_cols10(a[tr],target[tr],self.k);scaler=MaxAbsScaler().fit(a[tr][:,cols])
   est=LinearSVC(C=self.C,class_weight='balanced',dual='auto',max_iter=5000,random_state=42).fit(csr32(scaler.transform(a[tr][:,cols])),target[tr],sample_weight=sample_weight[tr])
   scores=est.decision_function(csr32(scaler.transform(a[va][:,cols])))
   assert np.array_equal(est.classes_,self.classes_);oof[va]=scores
  self.calibrators=[]
  for j,c in enumerate(self.classes_):
   cal=LogisticRegression(C=1.,class_weight='balanced' if self.balanced_calibration else None,max_iter=1000).fit(oof[:,j,None],(target==c).astype(int),sample_weight=sample_weight);self.calibrators.append(cal)
  self.cols=select_cols10(a,target,self.k);self.scaler=MaxAbsScaler().fit(a[:,self.cols]);self.est=LinearSVC(C=self.C,class_weight='balanced',dual='auto',max_iter=5000,random_state=42).fit(csr32(self.scaler.transform(a[:,self.cols])),target,sample_weight=sample_weight);self.coef_=self.est.coef_;return self
 def predict_proba(self,a):
  score=self.est.decision_function(csr32(self.scaler.transform(a[:,self.cols])))
  p=np.column_stack([cal.predict_proba(score[:,j,None])[:,1] for j,cal in enumerate(self.calibrators)])
  return p/np.maximum(p.sum(axis=1,keepdims=True),1e-12)

class Member10:
 def __init__(self,name,space,est,representation,numeric,cols=None,scaler=None):
  self.name=name;self.view='v10_'+representation;self.representation=representation;self.numeric=numeric;self.vec=space.compact_vec(np.arange(len(space.names)));self.est=est;self.cols=est.cols if cols is None else cols;self.raw_names=space.names[self.cols];self.names=self.raw_names;self.scaler=scaler
  lookup={g:i for i,g in enumerate(genes)};self.feature_gene_positions=np.array([lookup.get(str(s).split(':')[1],-1) if ':' in s and not str(s).startswith('numeric:') else -1 for s in space.names])
 def missing_matrix(self,packet):
  out=np.zeros((len(packet['events']),len(self.feature_gene_positions)),bool);valid=self.feature_gene_positions>=0
  out[:,valid]=packet['missing'][:,self.feature_gene_positions[valid]]
  return csr32(sparse.csr_matrix(out))
 def predict_packet(self,packet,records=None):
  rs=records10(packet,self.representation,self.numeric) if records is None else records;a=csr32(self.vec.transform(rs))
  if isinstance(self.est,MissingBernoulli10):return self.est.predict_proba(a,self.missing_matrix(packet))
  if isinstance(self.est,GroupSigmoidSVM10):return self.est.predict_proba(a)
  tx=a[:,self.cols]
  if self.scaler is not None:tx=csr32(self.scaler.transform(tx))
  return self.est.predict_proba(tx)
 def predict_frame(self,frame):return self.predict_packet(packet10(frame[genes]))

predict_members6=predict_members
def predict_members(members,weights,frame):
 old=[(m,w) for m,w in zip(members,weights) if not hasattr(m,'predict_frame')];p=predict_members6([m for m,w in old],[w for m,w in old],frame) if old else None
 packet=None
 for m,w in zip(members,weights):
  if hasattr(m,'predict_frame'):
   if packet is None:packet=packet10(frame[genes])
   q=m.predict_packet(packet)*w;p=q if p is None else p+q
 return p

C10={
 'gene_nb_a0.1':dict(rep='gene',algo='nb',alpha=.1,pooled=False,fit_prior=False,k=None,mask=0.),
 'gene_nb_a1':dict(rep='gene',algo='nb',alpha=1.,pooled=False,fit_prior=False,k=None,mask=0.),
 'gene_nb_empirical_prior':dict(rep='gene',algo='nb',alpha=.1,pooled=False,fit_prior=True,k=None,mask=0.),
 'gene_nb_pooled20':dict(rep='gene',algo='nb',alpha=20.,pooled=True,fit_prior=False,k=None,mask=0.),
 'type_nb_pooled20':dict(rep='type',algo='nb',alpha=20.,pooled=True,fit_prior=False,k=4096,mask=0.),
 'gene_lr':dict(rep='gene',algo='lr',C=.1,k=2048,mask=0.),
 'gene_lr_mask10':dict(rep='gene',algo='lr',C=.1,k=2048,mask=.1),
 'gene_lr_mask25':dict(rep='gene',algo='lr',C=.1,k=2048,mask=.25),
 'type_lr':dict(rep='type',algo='lr',C=.1,k=2048,mask=0.),
 'type_lr_mask10':dict(rep='type',algo='lr',C=.1,k=2048,mask=.1),
 'functional_lr':dict(rep='functional',algo='lr',C=.1,k=2048,mask=0.),
 'functional_lr_mask10':dict(rep='functional',algo='lr',C=.1,k=2048,mask=.1),
 'gene_svm':dict(rep='gene',algo='svm',C=.1,k=2048,mask=0.),
 'type_svm':dict(rep='type',algo='svm',C=.1,k=2048,mask=0.),
 'functional_svm':dict(rep='functional',algo='svm',C=.1,k=2048,mask=0.),
 'functional_svm_mask10':dict(rep='functional',algo='svm',C=.1,k=2048,mask=.1),
 'functional_lgb':dict(rep='functional',algo='lgb',k=2048,mask=0.),
 'functional_lgb_mask10':dict(rep='functional',algo='lgb',k=2048,mask=.1),
 'functional_lgb_mask25':dict(rep='functional',algo='lgb',k=2048,mask=.25)}

def prepare10(idx,representation,numeric,cache):
 key=(representation,numeric)
 if key not in cache:
  if '_clean' not in cache:cache['_clean']=packet10(X.iloc[idx],[events[i] for i in idx])
  rs=records10(cache['_clean'],representation,numeric);space=Space();a=space.fit(rs);cache[key]=(space,a)
 return cache[key]

def train10(name,idx,cache):
 cfg=C10[name];numeric=cfg['algo'] not in ['nb','cnb'];space,a=prepare10(idx,cfg['rep'],numeric,cache);target=yi[idx];rw=np.ones(len(idx));train_groups=groups[idx]
 augmented=None
 if cfg['mask']:
  key=('masked',cfg['mask'])
  if key not in cache:cache[key]=mask_packet10(cache['_clean'],cfg['mask'],1042+int(cfg['mask']*100))
  augmented=cache[key];ar=records10(augmented,cfg['rep'],numeric);a_aug=space.transform(ar);a=csr32(sparse.vstack([a,a_aug],format='csr'));target=np.r_[target,target];train_groups=np.r_[train_groups,train_groups];rw=np.r_[rw,.5*rw]/1.5
 if cfg['algo']=='nb':
  est=MissingBernoulli10(cfg['alpha'],cfg['pooled'],cfg['fit_prior'],cfg['k'])
  temp=Member10(name,space,est,cfg['rep'],False,cols=np.arange(a.shape[1]));missing=temp.missing_matrix(cache['_clean']);est.fit(a,target,missing,sample_weight=rw)
  return Member10(name,space,est,cfg['rep'],False)
 if cfg['algo']=='cnb':
  cols=select_cols10(a,target,cfg['k']);est=ComplementNB(alpha=cfg['alpha'],norm=False).fit(a[:,cols],target,sample_weight=rw)
  return Member10(name,space,est,cfg['rep'],False,cols=cols)
 if cfg['algo']=='svm':
  est=GroupSigmoidSVM10(cfg['C'],cfg['k'],cfg.get('balanced_calibration',False)).fit(a,target,train_groups,rw);return Member10(name,space,est,cfg['rep'],True)
 # Outer selection and scaler are fit on the original training patients only.
 cols=select_cols10(prepare10(idx,cfg['rep'],numeric,cache)[1],yi[idx],cfg['k'])
 scaler=MaxAbsScaler().fit(prepare10(idx,cfg['rep'],numeric,cache)[1][:,cols]) if cfg['algo']=='lr' else None
 tx=a[:,cols];tx=csr32(scaler.transform(tx)) if scaler is not None else csr32(tx)
 est=LogisticRegression(C=cfg['C'],class_weight='balanced',max_iter=2000) if cfg['algo']=='lr' else LGBMClassifier(n_estimators=300,learning_rate=.04,num_leaves=15,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=42)
 est.fit(tx,target,sample_weight=rw);return Member10(name,space,est,cfg['rep'],True,cols=cols,scaler=scaler)

# A missing feature must contribute no Bernoulli presence or absence evidence.
toy=sparse.csr_matrix([[1,0],[1,0],[0,1],[0,1]],dtype=np.float32);toy_m=sparse.csr_matrix(toy.shape,dtype=np.float32)
nbtoy=MissingBernoulli10(alpha=1.).fit(toy,np.array([0,0,1,1]),toy_m)
assert np.allclose(nbtoy.predict_proba(sparse.csr_matrix([[1,0]],dtype=np.float32),sparse.csr_matrix([[0,0]],dtype=np.float32)),[[.9,.1]])
assert np.allclose(nbtoy.predict_proba(sparse.csr_matrix([[0,0]],dtype=np.float32),sparse.csr_matrix([[1,1]],dtype=np.float32)),[[.5,.5]])
assert all(kind10(t)=='synonymous' for t in expand10('1_2FS>FS'))
(OUT/'configurations_v10.json').write_text(json.dumps(C10,indent=2));print('V10 candidates:',len(C10),'semantic checks passed')

C10.update({'functional_svm_balanced': {'rep': 'functional', 'algo': 'svm', 'C': 0.1, 'k': 2048, 'mask': 0.0, 'balanced_calibration': True}, 'functional_svm_balanced_mask10': {'rep': 'functional', 'algo': 'svm', 'C': 0.1, 'k': 2048, 'mask': 0.1, 'balanced_calibration': True}, 'type_complement_nb_a0.1': {'rep': 'type', 'algo': 'cnb', 'alpha': 0.1, 'k': 4096, 'mask': 0.0}, 'type_complement_nb_a1': {'rep': 'type', 'algo': 'cnb', 'alpha': 1.0, 'k': 4096, 'mask': 0.0}, 'functional_complement_nb_a1': {'rep': 'functional', 'algo': 'cnb', 'alpha': 1.0, 'k': 4096, 'mask': 0.0}})
(OUT/'configurations_v10.json').write_text(json.dumps(C10,indent=2))
print('Extended candidates:',len(C10))


V10 candidates: 19 semantic checks passed
Extended candidates: 24


In [8]:
from catboost import CatBoostClassifier
from sklearn.svm import SVC
from sklearn.decomposition import TruncatedSVD
from sklearn.neural_network import MLPClassifier
# Predeclared grid. Outer validation is never passed as eval_set or early stopping data.
GRID13={
 'cat128_d5':dict(algo='cat',k=128,depth=5,iterations=400,loss='MultiClass'),
 'cat256_d6':dict(algo='cat',k=256,depth=6,iterations=600,loss='MultiClass'),
 'cat128_ova':dict(algo='cat',k=128,depth=5,iterations=500,loss='MultiClassOneVsAll'),
 'lgb_cat256_l15':dict(algo='cat_lgb',k=256,leaves=15,minleaf=25),
 'lgb_cat512_l31':dict(algo='cat_lgb',k=512,leaves=31,minleaf=25),
 'lgb_cat256_regular':dict(algo='cat_lgb',k=256,leaves=15,minleaf=50),
 'lgb_dense63':dict(algo='dense_lgb',k=4096,leaves=63,minleaf=20),
 'rbf_svd128_c3':dict(algo='svm',k=4096,components=128,C=3.),
 'rbf_svd128_c10':dict(algo='svm',k=4096,components=128,C=10.),
 'mlp_256_64':dict(algo='mlp',k=4096,alpha=10.),
 'mlp_256_64_a1':dict(algo='mlp',k=4096,alpha=1.),
 'mlp_256_64_a0.1':dict(algo='mlp',k=4096,alpha=.1),
}
(OUT/'protocol.json').write_text(json.dumps({'grid':GRID13,'blend_new_weights':[.25,.5,.75,1.], 'primary_cv_seed':43,'confirmation_seed':313,'holdout_seed':42,'max_finalists':2,'test_policy':'never opened; audit hook enabled','goal_macro_f1':.60,'selection':'primary top individual and top overall blend, freeze; confirm then maximum confirm clean F1; never select using holdout','patient_specific_gating':False},indent=2))
# Fixed per-row normalization. No corpus-level information is learned here.
raw13=X.fillna('__MISSING__').astype(str).copy()
for g in genes:raw13[g]=raw13[g].map(lambda s:' '.join(sorted(set(canon(t) for t in s.split()))) if s not in ['WT','__MISSING__'] else s)
# Audit canonical profile collisions across the retained original development/holdout split.
canonical13=pd.util.hash_pandas_object(raw13,index=False).to_numpy()
collisions13=set(canonical13[dev]).intersection(canonical13[hold])
# If a new representation-identical profile crosses the holdout, remove its development rows.
# This does not inspect holdout labels and does not move patients into the training set.
if collisions13:
 excluded13=dev[np.isin(canonical13[dev],list(collisions13))];dev=dev[~np.isin(canonical13[dev],list(collisions13))]
else:excluded13=np.array([],int)
# Canonical groups are used for every new split; they include all identical raw profiles too.
groups13=canonical13
folds13=folds if len(excluded13)==0 and all(not set(groups13[dev[a]]).intersection(groups13[dev[b]]) for a,b in folds) else list(StratifiedGroupKFold(3,shuffle=True,random_state=43).split(X.iloc[dev],y[dev],groups13[dev]))
reuse_primary13=len(excluded13)==0 and all(np.array_equal(a,aa) and np.array_equal(b,bb) for (a,b),(aa,bb) in zip(folds13,folds))
assert not set(groups13[dev]).intersection(groups13[hold])
for a,b in folds13:assert not set(groups13[dev[a]]).intersection(groups13[dev[b]])
(OUT/'representation_collision_audit.json').write_text(json.dumps({'canonical_holdout_overlap_profiles':len(collisions13),'excluded_development_rows':excluded13.tolist(),'dev_n':len(dev),'holdout_n':len(hold)},indent=2))
# Numeric per-patient summaries and gene presence, computed without fitted statistics.
presence13=np.zeros((len(train),len(genes)),np.float32);numeric13=[];types13=pd.DataFrame('WT',index=X.index,columns=genes,dtype=str).mask(X.isna(),'__MISSING__')
lookup13={g:j for j,g in enumerate(genes)}
for i,row in enumerate(events):
 counts=Counter();n=0
 for g,ts in row:
  presence13[i,lookup13[g]]=1;ks=sorted(set(kind(t) for t in ts));types13.iat[i,lookup13[g]]='|'.join(ks);counts.update(kind(t) for t in ts);n+=len(ts)
 numeric13.append([np.log1p(len(row)),np.log1p(n),*[np.log1p(counts[t]) for t in ['substitution','synonymous','stop','frameshift','deletion','insertion','other']]])
numeric13=np.array(numeric13,np.float32)
class Category13:
 def fit(self,idx,k):
  score=np.nan_to_num(chi2(presence13[idx],yi[idx])[0],nan=0,posinf=0,neginf=0);self.cols=np.argsort(-score,kind='stable')[:k];self.genes=[genes[j] for j in self.cols]
  self.allowed={g:set(raw13.iloc[idx][g].value_counts().loc[lambda x:x>=3].index)|{'WT','__MISSING__'} for g in self.genes}
  return self
 def transform(self,idx=None,frame=None):
  if frame is None:raw=raw13.iloc[idx];typ=types13.iloc[idx];nums=numeric13[idx]
  else:
   raw=frame[genes].fillna('__MISSING__').astype(str).copy()
   for g in genes:raw[g]=raw[g].map(lambda s:' '.join(sorted(set(canon(t) for t in s.split()))) if s not in ['WT','__MISSING__'] else s)
   ev=parse_events(frame[genes]);typ=pd.DataFrame('WT',index=frame.index,columns=genes,dtype=str).mask(frame[genes].isna(),'__MISSING__');ns=[]
   for i,row in enumerate(ev):
    c=Counter();ne=0
    for g,ts in row:typ.iat[i,lookup13[g]]='|'.join(sorted(set(kind(t) for t in ts)));c.update(kind(t) for t in ts);ne+=len(ts)
    ns.append([np.log1p(len(row)),np.log1p(ne),*[np.log1p(c[t]) for t in ['substitution','synonymous','stop','frameshift','deletion','insertion','other']]])
   nums=np.array(ns,np.float32)
  result={}
  for g in self.genes:result[g]=raw[g].where(raw[g].isin(self.allowed[g]),'__MUT_OTHER__').to_numpy()
  for g in self.genes:result['type_'+g]=typ[g].to_numpy()
  for j in range(nums.shape[1]):result['num_'+str(j)]=nums[:,j]
  return pd.DataFrame(result)
class NewModel13:
 v13_export=True
 def __init__(self,name,cfg):self.name=name;self.cfg=cfg
 def fit(self,idx,cache):
  c=self.cfg;self.mode=c['algo'];target=yi[idx]
  if self.mode in ['cat','cat_lgb']:
   key=('cat',c['k'])
   if key not in cache:
    prep=Category13().fit(idx,c['k']);a=prep.transform(idx);cache[key]=(prep,a)
   self.prep,a=cache[key];self.ncat=2*c['k'];self.raw_names=np.array(a.columns)
   if self.mode=='cat':
    self.est=CatBoostClassifier(iterations=c['iterations'],depth=c['depth'],learning_rate=.045,loss_function=c['loss'],auto_class_weights='Balanced',l2_leaf_reg=10,random_strength=1,random_seed=42,thread_count=4,allow_writing_files=False,verbose=100,max_ctr_complexity=1,one_hot_max_size=255)
    self.est.fit(a,target,cat_features=list(range(self.ncat)))
   else:
    self.maps={col:{v:i for i,v in enumerate(a[col].unique())} for col in a.columns[:self.ncat]};aa=self.encode(a)
    self.est=LGBMClassifier(n_estimators=600,learning_rate=.035,num_leaves=c['leaves'],min_child_samples=c['minleaf'],class_weight='balanced',reg_lambda=20,colsample_bytree=.8,cat_smooth=20,cat_l2=10,min_data_per_group=15,max_cat_to_onehot=4,verbosity=-1,n_jobs=2,random_state=42)
    self.est.fit(aa,target,categorical_feature=list(range(self.ncat)))
  else:
   key=('raw_matrix',)
   if key not in cache:
    records=EvidenceEncoder(1.,1.).fit([events[i] for i in idx]).transform([events[i] for i in idx]);space=Space();a=space.fit(records);cols=select_cols10(a,target,c['k']);cache[key]=(space,a,cols)
   space,a,self.cols=cache[key];self.vec=space.compact_vec(np.arange(len(space.names)));self.raw_names=space.names[self.cols];a=a[:,self.cols]
   self.scaler=MaxAbsScaler().fit(a);a=csr32(self.scaler.transform(a))
   if self.mode=='svm':
    key=('svd',c['components'])
    if key not in cache:
     svd=TruncatedSVD(n_components=min(c['components'],a.shape[1]-1),random_state=42);dense=svd.fit_transform(a);scale=StandardScaler().fit(dense);cache[key]=(svd,scale,scale.transform(dense))
    self.svd,self.zscale,z=cache[key];self.est=SVC(C=c['C'],kernel='rbf',gamma='scale',class_weight='balanced',decision_function_shape='ovr',break_ties=True,cache_size=256).fit(z,target)
   elif self.mode=='mlp':
    self.est=MLPClassifier(hidden_layer_sizes=(256,64),alpha=c['alpha'],max_iter=300,early_stopping=False,random_state=42,learning_rate_init=.001,batch_size=128)
    self.est.fit(a,target,sample_weight=compute_sample_weight('balanced',target))
   else:
    self.est=LGBMClassifier(n_estimators=700,learning_rate=.025,num_leaves=c['leaves'],min_child_samples=c['minleaf'],class_weight='balanced',reg_lambda=20,colsample_bytree=.7,verbosity=-1,n_jobs=2,random_state=42).fit(a,target)
  return self
 def encode(self,a):
  b=a.copy()
  for col,lookup in self.maps.items():b[col]=b[col].map(lookup).fillna(-1).astype(np.int32)
  return b
 def predict(self,idx=None,frame=None):
  if self.mode in ['cat','cat_lgb']:
   a=self.prep.transform(idx,frame)
   if self.mode=='cat_lgb':a=self.encode(a)
  else:
   ev=[events[i] for i in idx] if frame is None else parse_events(frame[genes]);a=self.vec.transform(EvidenceEncoder(1.,1.).fit([]).transform(ev));a=self.scaler.transform(a[:,self.cols])
   if self.mode=='svm':a=self.zscale.transform(self.svd.transform(a))
  if self.mode=='svm':p=softmax(self.est.decision_function(a),axis=1)
  else:
   p=np.asarray(self.est.predict_proba(a));p=p/np.maximum(p.sum(1,keepdims=True),1e-12)
  return p
print('V13 train-only models:',len(GRID13),'development',len(dev),'holdout',len(hold),flush=True)
# Compact inference-only wrappers deliberately exclude training DataFrames and fit methods.
class CatTransform13:
 def __init__(self,prep):self.genes=list(genes);self.selected=list(prep.genes);self.allowed=prep.allowed
 def transform(self,frame):
  raw=frame[self.selected].fillna('__MISSING__').astype(str).copy()
  for g in self.selected:raw[g]=raw[g].map(lambda s:' '.join(sorted(set(canon(t) for t in s.split()))) if s not in ['WT','__MISSING__'] else s)
  ev=parse_events(frame[self.genes]);values={g:raw[g].where(raw[g].isin(self.allowed[g]),'__MUT_OTHER__').to_numpy() for g in self.selected};types={g:[] for g in self.selected};nums=[];selected=set(self.selected)
  missing=frame[self.selected].isna().to_numpy()
  for i,row in enumerate(ev):
   known={g:ts for g,ts in row if g in selected};c=Counter();ne=0
   for g,ts in row:c.update(kind(t) for t in ts);ne+=len(ts)
   for j,g in enumerate(self.selected):types[g].append('|'.join(sorted(set(kind(t) for t in known[g]))) if g in known else ('__MISSING__' if missing[i,j] else 'WT'))
   nums.append([np.log1p(len(row)),np.log1p(ne),*[np.log1p(c[t]) for t in ['substitution','synonymous','stop','frameshift','deletion','insertion','other']]])
  for g in self.selected:values['type_'+g]=types[g]
  nums=np.asarray(nums,np.float32)
  for j in range(nums.shape[1]):values['num_'+str(j)]=nums[:,j]
  return pd.DataFrame(values)
class ExportModel13:
 v13_export=True
 def __init__(self,m):
  self.name=m.name;self.mode=m.mode;self.est=m.est;self.raw_names=m.raw_names;self.genes=list(genes)
  if m.mode in ['cat','cat_lgb']:
   self.prep=CatTransform13(m.prep);self.ncat=m.ncat
   if m.mode=='cat_lgb':self.maps=m.maps
  else:
   self.vec=m.vec;self.cols=m.cols;self.scaler=m.scaler
   if m.mode=='svm':self.svd=m.svd;self.zscale=m.zscale
 def predict(self,frame):
  if self.mode in ['cat','cat_lgb']:
   a=self.prep.transform(frame)
   if self.mode=='cat_lgb':
    for col,lookup in self.maps.items():a[col]=a[col].map(lookup).fillna(-1).astype(np.int32)
  else:
   a=self.vec.transform(EvidenceEncoder(1.,1.).fit([]).transform(parse_events(frame[self.genes])));a=self.scaler.transform(a[:,self.cols])
   if self.mode=='svm':a=self.zscale.transform(self.svd.transform(a))
  p=softmax(self.est.decision_function(a),axis=1) if self.mode=='svm' else np.asarray(self.est.predict_proba(a));return p/np.maximum(p.sum(1,keepdims=True),1e-12)

SOURCE_SHA13='d5fff1f19e2e7744c65fd1cc65c7faecb427e2be64c309ee90f7a0d61f032360'


V13 train-only models: 12 development 4960 holdout 1241


In [9]:
# Validate train identity and saved train-only baseline caches; never open any inference CSV.
reference13=None
v9=WORK/'artifacts/v9'
if reuse_primary13 and (v9/'component_oof.npz').exists():
 proof=json.loads((v9/'verification.json').read_text());assert proof['data_sha256']['train.csv']==hashlib.sha256((DATA/'train.csv').read_bytes()).hexdigest()
 q=np.load(v9/'component_oof.npz',allow_pickle=True);assert np.array_equal(q['dev'],dev) and np.array_equal(q['labels'],labels)
 reference13=np.tensordot(np.array([.0625,.0625,.0625,.140625,.140625,.140625,.140625,.25]),q['probabilities'][:,0],axes=(0,0))
P13={name:np.zeros((len(dev),NC),np.float32) for name in GRID13};baseline13=np.zeros((len(dev),NC),np.float32) if reference13 is None else reference13.copy();cvrows13=[]
completed13=[];checkpoint13=OUT/'oof_progress.npz'
if checkpoint13.exists():
 qc=np.load(checkpoint13,allow_pickle=True);assert np.array_equal(qc['dev'],dev) and np.array_equal(qc['labels'],labels) and str(qc['source_sha'])==SOURCE_SHA13
 P13={name:qc[name].copy() if name in qc.files else np.zeros((len(dev),NC),np.float32) for name in GRID13};completed13=qc['completed'].tolist();baseline13=qc['baseline'].copy();cvrows13=pd.read_csv(OUT/'cv_progress.csv').to_dict('records')
 print('Resuming completed model folds',completed13,flush=True)
for f,(a,b) in enumerate(folds13):
 idx=dev[a];va=dev[b];cache={}
 if reference13 is None:
  ms,ws=fit_v6(idx);baseline13[b]=predict_members(ms,ws,X.iloc[va]);del ms
 for name,cfg in GRID13.items():
  if f'{f}:{name}' in completed13:continue
  t=time.time();m=NewModel13(name,cfg).fit(idx,cache);p=m.predict(va);P13[name][b]=p
  cvrows13.append(dict(fold=f,model=name,macro_f1=f1_score(yi[va],p.argmax(1),average='macro',zero_division=0),accuracy=accuracy_score(yi[va],p.argmax(1)),seconds=time.time()-t))
  completed13.append(f'{f}:{name}');np.savez_compressed(checkpoint13,source_sha=SOURCE_SHA13,dev=dev,labels=labels,completed=np.array(completed13),baseline=baseline13,**P13)
  pd.DataFrame(cvrows13).to_csv(OUT/'cv_progress.csv',index=False);print('V13 CV',f,name,'F1',round(cvrows13[-1]['macro_f1'],4),'seconds',round(time.time()-t,1),flush=True);del m
 np.savez_compressed(OUT/'oof_probabilities.npz',dev=dev,labels=labels,baseline=baseline13,**P13)

def score13(prob,splits):return float(np.mean([f1_score(yi[dev[b]],prob[b].argmax(1),average='macro',zero_division=0) for a,b in splits]))
plans13={'v6_reference':[('v6_reference',1.)]};prob13={'v6_reference':baseline13,**P13}
for name in GRID13:
 for w in [.25,.5,.75,1.]:plans13[f'{name}_w{w}']=[('v6_reference',1-w),(name,w)]
individuals13=sorted(GRID13,key=lambda name:score13(P13[name],folds13),reverse=True)
# One predeclared equal blend of the two highest individual learners, then fixed blend weights.
for w in [.25,.5,.75,1.]:plans13[f'top2_new_w{w}']=[('v6_reference',1-w),(individuals13[0],w/2),(individuals13[1],w/2)]
rank13=pd.DataFrame([{'plan':name,'cv_macro_f1':score13(sum(w*prob13[member] for member,w in plan),folds13)} for name,plan in plans13.items()]).sort_values('cv_macro_f1',ascending=False)
rank13.to_csv(OUT/'model_comparison.csv',index=False);(OUT/'plans.json').write_text(json.dumps(plans13,indent=2))
best_individual=f'{individuals13[0]}_w1.0';finalists13=list(dict.fromkeys([rank13.iloc[0]['plan'],best_individual]));(OUT/'frozen_finalists.json').write_text(json.dumps(finalists13));print('FROZEN V13',finalists13,rank13.head().to_dict('records'),flush=True)
# New confirmation split; same development patients, not an independent external test.
confirm_seed=313;splits13=list(StratifiedGroupKFold(3,shuffle=True,random_state=confirm_seed).split(X.iloc[dev],y[dev],groups13[dev]));needed13={n for p in finalists13 for n,w in plans13[p] if w>0};needed13.add('v6_reference');C13={n:np.zeros((len(dev),NC),np.float32) for n in needed13};confirmrows13=[]
for f,(a,b) in enumerate(splits13):
 tr=dev[a];va=dev[b];cache={}
 for name in sorted(needed13):
  if name=='v6_reference':ms,ws=fit_v6(tr);C13[name][b]=predict_members(ms,ws,X.iloc[va]);del ms
  else:m=NewModel13(name,GRID13[name]).fit(tr,cache);C13[name][b]=m.predict(va);del m
  print('V13 confirm',f,name,'complete',flush=True)
 for name in ['v6_reference',*finalists13]:
  plan=plans13[name];p=sum(w*C13[n] for n,w in plan if w>0)
  confirmrows13.append(dict(fold=f,plan=name,macro_f1=f1_score(yi[va],p[b].argmax(1),average='macro',zero_division=0)))
 pd.DataFrame(confirmrows13).to_csv(OUT/'confirmation_cv.csv',index=False);np.savez_compressed(OUT/'confirmation_probabilities.npz',dev=dev,labels=labels,**C13)
confirmation13=pd.DataFrame(confirmrows13).groupby('plan').macro_f1.mean();winner13=max(finalists13,key=lambda name:confirmation13[name]);plan13=plans13[winner13]
# Freeze before opening any holdout model score. Keep the baseline if confirmation gains are absent.
accepted13=bool(confirmation13[winner13]>=confirmation13['v6_reference']+.003)
selection13={'winner':winner13,'plan':plan13,'confirmation_macro_f1':float(confirmation13[winner13]),'baseline_confirmation_f1':float(confirmation13['v6_reference']),'confirmation_gain_gate':.003,'accepted_candidate':accepted13,'target_macro_f1':.60,'test_access_attempts':forbidden_attempts13,'public_v10_user_reported':.3493803489}
(OUT/'selection.json').write_text(json.dumps(selection13,indent=2));print('V13 FROZEN WINNER',selection13,flush=True)

def fit_members13(idx,plan):
 models=[];weights=[];cache={}
 for name,w in plan:
  if w==0:continue
  if name=='v6_reference':old,ow=fit_v6(idx);models.extend(old);weights.extend(ow*w)
  else:models.append(NewModel13(name,GRID13[name]).fit(idx,cache));weights.append(w)
 return models,np.array(weights)
def predict13(bundle,frame):
 out=np.zeros((len(frame),NC),np.float64)
 for m,w in zip(bundle['models'],bundle['weights']):
  p=m.predict(frame=frame) if getattr(m,'v13_export',False) else predict_members([m],np.array([1.]),frame)
  out+=w*p
 return out/out.sum(1,keepdims=True)
# Evaluate exactly the frozen candidate, including if it fails the confirmation gate; never retune.
hm13,hw13=fit_members13(dev,plan13);hbundle13={'models':hm13,'weights':hw13};hp13=predict13(hbundle13,X.iloc[hold]);hpred13=hp13.argmax(1)
holdmetrics13={'macro_f1':float(f1_score(yi[hold],hpred13,average='macro',zero_division=0)),'accuracy':float(accuracy_score(yi[hold],hpred13)),'n':len(hold),'target_met':bool(f1_score(yi[hold],hpred13,average='macro')>=.60),'previously_observed_holdout':True}
(OUT/'holdout_metrics.json').write_text(json.dumps(holdmetrics13,indent=2));np.savez_compressed(OUT/'holdout_probabilities.npz',probabilities=hp13,indices=hold)
pd.DataFrame(classification_report(yi[hold],hpred13,target_names=labels,output_dict=True,zero_division=0)).T.to_csv(OUT/'holdout_classification_report.csv');print('V13 HOLDOUT',holdmetrics13,flush=True)
# No inference dataset or sample submission is opened. Only train is used for final fit.
models13,weights13=fit_members13(np.arange(len(train)),plan13);models13=[ExportModel13(m) if getattr(m,'v13_export',False) else m for m in models13];bundle13={'version':13,'models':models13,'weights':weights13,'genes':genes,'labels':labels,'classes':labels,'predict':predict13,'selection':selection13,'holdout':holdmetrics13}
with (OUT/'cancer_model_v13_experimental.pkl').open('wb') as f:cloudpickle.dump(bundle13,f)
with (OUT/'cancer_model_v13_experimental.pkl').open('rb') as f:reload13=cloudpickle.load(f)
probe13=X.iloc[dev[:19]];assert np.allclose(predict13(reload13,probe13),predict13(bundle13,probe13));assert np.allclose(predict13(reload13,probe13.iloc[:7]),predict13(bundle13,probe13)[:7])
report13='# v13 — train 전용 범주형·비선형 모델 비교\n\n목표는 기존 홀드아웃 Macro F1 0.60이며 달성을 보장하지 않습니다. 이번 실행은 test와 sample 파일을 열지 못하도록 audit hook으로 차단했습니다. 이전 버전의 test 관찰과 반복된 홀드아웃 사용의 설계 편향은 없어지지 않습니다.\n\n'+json.dumps(selection13,ensure_ascii=False,indent=2)+'\n\n## 교차검증\n\n'+rank13.to_markdown(index=False)+'\n\n## 고정 후보 추가 분할 검증\n\n'+pd.DataFrame(confirmrows13).to_markdown(index=False)+'\n\n## 후보 고정 후 기존 홀드아웃\n\n'+json.dumps(holdmetrics13,indent=2)+'\n\n외부 검증을 조기 종료에 쓰지 않았습니다. 모든 선별·빈도·스케일·SVD는 학습 fold에서만 적합했습니다. 표기 정규화 후 동일 프로필 그룹이 분할을 넘지 않게 점검하고 충돌하는 개발 환자는 제외했습니다. 최종 모델은 전체 train 학습 후 별도 실험용 pickle로 저장했으며 제출 CSV는 생성하지 않았습니다. 주 모델은 자동 교체하지 않았습니다.\n'
(WORK/'analysis_report_v13.md').write_text(report13)
(OUT/'verification.json').write_text(json.dumps({'status':'passed','version':13,'train_sha256':hashlib.sha256((DATA/'train.csv').read_bytes()).hexdigest(),'forbidden_inference_file_accesses':forbidden_attempts13,'holdout':holdmetrics13,'selection':selection13,'same_process_pickle_reload':'passed','goal_achieved':holdmetrics13['target_met'],'test_opened':False,'sample_opened':False},indent=2));print('V13 DONE',flush=True)


Resuming completed model folds ['0:cat128_d5', '0:cat256_d6', '0:cat128_ova', '0:lgb_cat256_l15', '0:lgb_cat512_l31', '0:lgb_cat256_regular', '0:lgb_dense63', '0:rbf_svd128_c3', '0:rbf_svd128_c10', '0:mlp_256_64', '1:cat128_d5', '1:cat256_d6', '1:cat128_ova', '1:lgb_cat256_l15']
V13 CV 0 mlp_256_64_a1 F1 0.3628 seconds 17.8
V13 CV 0 mlp_256_64_a0.1 F1 0.3423 seconds 11.6
V13 CV 1 lgb_cat512_l31 F1 0.3936 seconds 104.0
V13 CV 1 lgb_cat256_regular F1 0.3829 seconds 53.0
V13 CV 1 lgb_dense63 F1 0.3815 seconds 32.5
V13 CV 1 rbf_svd128_c3 F1 0.3065 seconds 1.8
V13 CV 1 rbf_svd128_c10 F1 0.3008 seconds 1.7
V13 CV 1 mlp_256_64 F1 0.0021 seconds 6.9
V13 CV 1 mlp_256_64_a1 F1 0.3589 seconds 10.7
V13 CV 1 mlp_256_64_a0.1 F1 0.3395 seconds 11.0
0:	learn: 3.2188843	total: 142ms	remaining: 56.8s
100:	learn: 2.4981669	total: 8.59s	remaining: 25.4s
200:	learn: 2.2349034	total: 17.2s	remaining: 17s
300:	learn: 2.0008939	total: 25.6s	remaining: 8.42s
399:	learn: 1.9079029	total: 34s	remaining: 0us
V13 

/Users/mac/project/cancer_project/.venv-catboost/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0 0 0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv-catboost/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv-catboost/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv-catboost/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0 0] are constant.
  warnings.warn("Features %s are constant." % 

## 선택 실행: 사용자 승인 최종 제출 추론
이 셀부터는 test/sample을 **최종 추론에만** 사용합니다. 앞선 문서·지표의 test 미사용은 학습·선별·검증 단계에 대한 기록입니다. 이 셀은 재학습·튜닝하지 않고 저장 모델로 제출 CSV를 만듭니다. 연구용 open guard는 유지하고 별도 추론 프로세스를 사용합니다. 기존 제출 파일과 결과가 다르면 덮어쓰지 않습니다. 저장 모델이 있으면 이 셀만 별도 실행할 수 있습니다. v18 파일은 스트레스 조건 성능 저하가 있는 후속 실험용이며, v19와 구분하세요.

In [ ]:
# 사용자 승인 최종 제출 추론: 학습/검증 단계와 구분합니다.
from pathlib import Path
import subprocess,sys
roots_export=[Path.cwd(),*Path.cwd().parents]
repo_export=next((p for p in roots_export if (p/'data'/'test.csv').is_file()),None)
if repo_export is None:raise FileNotFoundError('공식 data/test.csv와 sample_submission.csv가 필요합니다.')
work_export=repo_export/'models'/'혜진' if (repo_export/'models'/'혜진').is_dir() else repo_export
model_export=work_export/'artifacts'/'v13'/'cancer_model_v13_experimental.pkl'
if not model_export.is_file():raise FileNotFoundError('앞선 학습 셀에서 저장한 모델이 필요합니다.')
# 연구 단계 open guard가 설치된 부모와 구분해 새 추론 프로세스에서만 test를 읽습니다.
# fit/선별/파라미터 변경 없이 저장 모델의 predict만 호출합니다.
inference_program="from pathlib import Path\nimport sys,json,cloudpickle,numpy as np,pandas as pd,hashlib\nfrom threadpoolctl import threadpool_limits\nthreadpool_limits(2)\nmodelpath,testpath,samplepath,outpath,labelpath=map(Path,sys.argv[1:6])\nwith modelpath.open('rb') as f:model=cloudpickle.load(f)\ntest=pd.read_csv(testpath);sample=pd.read_csv(samplepath)\nassert list(sample.columns)==['ID','SUBCLASS'] and test.ID.is_unique and sample.ID.is_unique and set(test.ID)==set(sample.ID)\nx=test.drop(columns=['ID'])\nif isinstance(model,dict):\n classes=np.asarray(model['classes']);predict=lambda frame:model['predict'](model,frame)\nelse:\n with np.load(labelpath,allow_pickle=True) as q:classes=np.asarray(q['labels']).astype(str)\n predict=model.predict_frame\np=np.asarray(predict(x));assert p.shape==(len(test),len(classes)) and len(classes)==26 and np.isfinite(p).all() and (p>=-1e-8).all() and np.allclose(p.sum(1),1,atol=1e-5)\nassert np.allclose(predict(x.iloc[:8]),p[:8],atol=1e-6)\nresult=sample.copy();result.SUBCLASS=result.ID.map(pd.Series(classes[p.argmax(1)],index=test.ID))\nassert result.ID.equals(sample.ID) and not result.isna().any().any()\npayload=result.to_csv(index=False).encode('utf-8');outpath.parent.mkdir(exist_ok=True,parents=True)\nif outpath.exists():\n assert outpath.read_bytes()==payload,'기존 제출 파일과 다릅니다. 파일을 확인하고 별도 이름을 지정하세요.'\nelse:outpath.write_bytes(payload)\nassert pd.read_csv(outpath).equals(result)\nproof={'rows':len(result),'test_use':'inference only','training_performed':False,'sample_id_order':'verified','probabilities':'verified','batch_invariance':'passed','model_sha256':hashlib.sha256(modelpath.read_bytes()).hexdigest(),'csv_sha256':hashlib.sha256(payload).hexdigest()}\nproofpath=modelpath.parent/'submission_inference_verification.json';proofpath.write_text(json.dumps(proof,ensure_ascii=False,indent=2));print(str(outpath));print(json.dumps(proof,ensure_ascii=False,indent=2))\n"
result_export=subprocess.run([sys.executable,'-c',inference_program,str(model_export.resolve()),str((repo_export/'data/test.csv').resolve()),str((repo_export/'data/sample_submission.csv').resolve()),str((repo_export/'submission'/'혜진_submission_v13_experimental.csv').resolve()),str((work_export/'artifacts/v13/oof_probabilities.npz').resolve())],capture_output=True,text=True)
if result_export.returncode:raise RuntimeError(result_export.stderr)
print(result_export.stdout)
